# kinfast on a GPU

Any CUDA runtime works. On Colab that is Runtime > Change runtime
type; a free T4 is enough for every number here.

A bigger card changes one thing and not the other. Batched forward
kinematics is bound by memory bandwidth and kernel launches rather
than arithmetic, so an L4 lands near a T4 despite four times the
FLOPs, while an A100 is genuinely faster because it has roughly five
times the bandwidth. What a big card really buys is headroom: the
benchmark grows the batch by powers of ten until the card fills, so
40 or 80 GB reaches batch sizes a 16 GB card cannot hold.

Three steps, in this order:

1. check that every module agrees with the CPU on CUDA
2. measure throughput and write `BENCHMARK_GPU.md`
3. render the demo gif

Run every cell top to bottom, then bring back the table and the gif.


## 1. Setup

If the GPU check below prints `no CUDA device`, the runtime is still on CPU:
switch it in Runtime > Change runtime type and run the cell again.

In [ ]:
import torch
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no CUDA device")

In [ ]:
import os, sys, subprocess, importlib

REPO = "/content/kinfast"
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/VihanAggarwal/kinfast", REPO], check=True)
os.chdir(REPO)

# torch already ships with CUDA on Colab, so do not let pip replace it
!pip install -q -e . --no-deps
!pip install -q pytest mujoco pytorch-kinematics xacro hypothesis

# An editable install writes the source location into a .pth file, and .pth
# files are only read when the interpreter starts. In a kernel that is already
# running it therefore has no effect. Colab also keeps /content on sys.path,
# which leaves the bare clone directory importable as an empty namespace
# package, so `import kinfast` succeeds and gives back nothing. Point at the
# source tree directly and drop anything already bound to the name.
src = os.path.join(REPO, "src")
if src not in sys.path:
    sys.path.insert(0, src)
sys.modules.pop("kinfast", None)
importlib.invalidate_caches()

import kinfast
print("kinfast", kinfast.__version__, "from", os.path.dirname(kinfast.__file__))


In [ ]:
# the robot files are fetched, not vendored, so grab them
!python examples/gallery.py --fetch 2>&1 | tail -3
!ls examples/assets/gallery | head

## 2. Correctness first

Every module has to agree with the CPU path before any number is worth
measuring. These tests skip themselves without CUDA, so on a GPU runtime they
should all run and pass.

In [ ]:
!python -m pytest tests/test_gpu.py -v

## 3. The benchmark

Batches grow by powers of ten until the card runs out of memory, with explicit
synchronization around every timed call so a kernel launch is never mistaken
for finished work. The result lands in `examples/assets/BENCHMARK_GPU.md`.

In [ ]:
!python examples/gpu_benchmark.py

## 4. The 10,000 arm demo

This is the gif for the README: ten thousand IK problems solved in one batch.

In [ ]:
!python examples/demo_10k_arms.py --urdf examples/assets/gallery/panda.urdf \
    --n 10000 --restarts 4 --gif demo_gpu.gif
from IPython.display import Image, display
display(Image("demo_gpu.gif"))

## 5. What to bring home

Two artifacts: the benchmark table below, and `demo_gpu.gif` from the file
browser on the left (right click > Download). Paste the table into the
README's speed section and commit the gif.

In [ ]:
print(open("examples/assets/BENCHMARK_GPU.md").read())

In [ ]:
# a one line summary worth keeping next to the table
import time, torch, kinfast
robot = kinfast.load("examples/assets/gallery/panda.urdf").to("cuda")
q = robot.random_configs(100_000)
robot.fk_all(q); torch.cuda.synchronize()
t0 = time.perf_counter(); robot.fk_all(q); torch.cuda.synchronize()
dt = time.perf_counter() - t0
print(f"{torch.cuda.get_device_name(0)}: forward kinematics for 100,000 "
      f"configurations in {dt*1e3:.1f} ms ({100_000/dt:,.0f} per second)")

## How far does this card go?

Grows the batch until CUDA refuses, then reports the largest one
that fit. On a 40 GB A100 this reaches into the millions.


In [ ]:
import time, torch, kinfast
robot = kinfast.load('examples/assets/gallery/panda.urdf').to('cuda')
n, best = 100_000, None
while n <= 50_000_000:
    try:
        torch.cuda.reset_peak_memory_stats()
        q = robot.random_configs(n)
        robot.fk_all(q); torch.cuda.synchronize()
        t0 = time.perf_counter(); robot.fk_all(q); torch.cuda.synchronize()
        dt = time.perf_counter() - t0
        peak = torch.cuda.max_memory_allocated() / 2**30
        best = (n, dt, peak)
        print(f'{n:>12,} configs  {dt*1e3:8.1f} ms  {n/dt:>14,.0f}/s  {peak:5.1f} GB')
        del q; torch.cuda.empty_cache()
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f'{n:>12,} configs  out of memory')
        break
    n *= 10
if best:
    n, dt, peak = best
    print(f'headline: forward kinematics for {n:,} robot configurations '
          f'in one call, {dt*1e3:.0f} ms, {peak:.1f} GB, on a '
          f'{torch.cuda.get_device_name(0)}')
